# 4.1 — Clasificacion multiclase con clusters malignos no supervisados

Replica simplificada del notebook 4.0, usando los datos enriquecidos con
etiquetas de cluster (`mal_cluster`) generados en el notebook 1.2.

**Cambio clave**: los 18 tipos de cancer se reemplazan por k clusters malignos
no supervisados, reduciendo el problema a k+1 clases (k clusters + nonMalignant).

Ranking:
1. Minimizar `test_cancer_fn`
2. Minimizar `test_cancer_fnr`
3. Maximizar `test_f1_macro`

In [1]:
from __future__ import annotations
from pathlib import Path
from dataclasses import replace
import logging
import warnings
from sklearn.exceptions import ConvergenceWarning

import pandas as pd
import numpy as np

from time import perf_counter
from tqdm.auto import tqdm

from genomics_dl.models.train_multiclass import MulticlassTrainConfig, run_training

warnings.filterwarnings("ignore", category=ConvergenceWarning)
logging.getLogger("alembic").setLevel(logging.ERROR)
logging.getLogger("alembic.runtime.migration").setLevel(logging.ERROR)
logging.getLogger("mlflow").setLevel(logging.ERROR)
logging.getLogger("sqlalchemy").setLevel(logging.ERROR)
warnings.filterwarnings(
    "ignore", category=RuntimeWarning,
    message=".*invalid value encountered in divide.*",
)
warnings.filterwarnings(
    "ignore", category=RuntimeWarning,
    message=".*A worker stopped while some jobs were given to the executor.*",
)

/workspaces/TFM/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Rutas y carga de datos (con clusters)

In [2]:
DATA_PROCESSED = Path("../data/processed")
TRAIN_PATH = DATA_PROCESSED / "gse183635_tep_tpm_train_clustered.parquet"
TEST_PATH  = DATA_PROCESSED / "gse183635_tep_tpm_test_clustered.parquet"

df_train = pd.read_parquet(TRAIN_PATH)
df_test  = pd.read_parquet(TEST_PATH)

df_train.shape, df_test.shape

((1880, 5453), (471, 5453))

### Separacion genes vs metadatos

In [3]:
metadata_cols = [
    "Sample ID", "Patient_group", "Stage", "Sex", "Age",
    "Sample-supplying institution", "Training series",
    "Evaluation series", "Validation series", "lib.size",
    "classificationScoreCancer", "Class_group", "mal_cluster",
]

gene_cols = [c for c in df_train.columns if str(c).startswith("ENSG")]

assert "mal_cluster" in df_train.columns, "Ejecutar notebook 1.2 primero"
assert len(gene_cols) > 0

len(gene_cols), gene_cols[:5]

(5440,
 ['ENSG00000000419',
  'ENSG00000000460',
  'ENSG00000000938',
  'ENSG00000001036',
  'ENSG00000001461'])

### Sanity check: distribucion de clases con clusters malignos

In [4]:
from genomics_dl.models.train_multiclass import build_y_multiclass_with_malignant_clusters

y_train_clust = build_y_multiclass_with_malignant_clusters(
    df_train,
    class_group_col="Class_group",
    patient_group_col="Patient_group",
    cluster_col="mal_cluster",
)

n_classes = len(np.unique(y_train_clust))
print(f"Clases unicas: {n_classes}")
print(pd.Series(y_train_clust).value_counts())

Clases unicas: 3
Malignant_0     867
nonMalignant    578
Malignant_1     435
Name: count, dtype: int64


## Sweep simplificado (16 configuraciones)

In [5]:
def slugify_token(value):
    return str(value).replace(".", "p").replace("-", "m")

def build_model_name(clf_name, feat_cfg, malignant_weight, variant_tag):
    return "_".join([
        clf_name,
        f"pca{int(feat_cfg['use_pca'])}",
        f"log{int(feat_cfg['selector_on_log'])}",
        f"vq{int(feat_cfg['var_quantile']*100)}",
        f"mw{slugify_token(malignant_weight)}",
        variant_tag,
    ])

def fmt_secs(s: float) -> str:
    s = int(max(0, s))
    h = s // 3600
    m = (s % 3600) // 60
    ss = s % 60
    if h > 0:
        return f"{h:d}h {m:02d}m {ss:02d}s"
    if m > 0:
        return f"{m:d}m {ss:02d}s"
    return f"{ss:d}s"

In [6]:
base_cfg = MulticlassTrainConfig(
    train_path=str(TRAIN_PATH),
    test_path=str(TEST_PATH),
    model_name="multiclass_mal_clustered",
    model_version="v0.1.0",
    use_pca=False,
    var_quantile=0.2,
    selector_on_log=False,
    pca_var_threshold=0.9,
    cv_splits=5,
    min_cancer_recall_for_threshold=0.9,
    threshold_objective="specificity",
    experiment_name="gse183635_multiclass_mal_clustered",
    save_local_bundle=False,
    save_plots=False,
    malignant_cluster_col="mal_cluster",  # activa clusters malignos
)

# Grid simplificado basado en mejores configuraciones de 4.0
feat_grid = [
    dict(use_pca=False, selector_on_log=False, var_quantile=0.15, pca_var_threshold=0.9),
    dict(use_pca=False, selector_on_log=False, var_quantile=0.20, pca_var_threshold=0.9),
]

clf_grid = [
    ("rf", dict(n_estimators=500, max_depth=None)),
    ("rf", dict(n_estimators=1000, max_depth=None)),
    ("extratrees", dict(n_estimators=1000, max_depth=None, min_samples_leaf=2)),
    ("logreg", dict(solver="lbfgs", max_iter=8500, C=1.0)),
]

malignant_weights = [3.0, 6.0]

sweep = []
for feat_cfg in feat_grid:
    for clf_name, clf_params in clf_grid:
        for mw in malignant_weights:
            sweep.append(dict(
                feat_cfg=feat_cfg, clf_name=clf_name,
                clf_params=clf_params, mw=mw,
            ))

print(f"Total combinaciones: {len(sweep)}")

Total combinaciones: 16


In [7]:
results = []
errors = []

total = len(sweep)
start_all = perf_counter()

ema = None
alpha = 0.25
done = 0

pbar = tqdm(sweep, total=total, desc="Sweep mal_clustered", unit="run")

for combo in pbar:
    t0 = perf_counter()

    feat_cfg = combo["feat_cfg"]
    clf_name = combo["clf_name"]
    clf_params = combo["clf_params"]
    mw = combo["mw"]

    model_name = build_model_name(clf_name, feat_cfg, mw, variant_tag="malclust")
    cfg = replace(
        base_cfg,
        model_name=model_name,
        clf_name=clf_name,
        clf_params=clf_params,
        malignant_weight=mw,
        use_pca=feat_cfg["use_pca"],
        selector_on_log=feat_cfg["selector_on_log"],
        var_quantile=feat_cfg["var_quantile"],
        pca_var_threshold=feat_cfg["pca_var_threshold"],
    )

    try:
        out = run_training(cfg, feature_cols=gene_cols)
        tm = out["test_metrics"]
        results.append({
            "model_name": model_name,
            "clf_name": clf_name,
            "mw": mw,
            "var_quantile": feat_cfg["var_quantile"],
            "test_cancer_fn": tm["cancer_fn"],
            "test_cancer_fnr": tm["cancer_fnr"],
            "test_cancer_recall": tm["cancer_recall_sensitivity"],
            "test_cancer_specificity": tm["cancer_specificity"],
            "test_f1_macro": tm["f1_macro"],
            "test_accuracy": tm["accuracy"],
            "test_balanced_accuracy": tm["balanced_accuracy"],
            "test_cancer_roc_auc": tm["cancer_roc_auc"],
            "test_cancer_pr_auc": tm["cancer_pr_auc"],
            "mlflow_run_id": out["mlflow_run_id"],
        })
    except Exception as e:
        errors.append({
            "model_name": model_name,
            "clf_name": clf_name,
            "mw": mw,
            "var_quantile": feat_cfg["var_quantile"],
            "error": repr(e),
        })

    dt = perf_counter() - t0
    ema = dt if ema is None else (alpha * dt + (1 - alpha) * ema)

    done += 1
    elapsed = perf_counter() - start_all
    remaining = (total - done) * (ema if ema is not None else 0.0)

    pbar.set_postfix({
        "last": fmt_secs(dt), "avg": fmt_secs(ema),
        "elapsed": fmt_secs(elapsed), "eta": fmt_secs(remaining),
        "ok": len(results), "err": len(errors),
    })

res_df = (
    pd.DataFrame(results)
      .sort_values(["test_cancer_fn", "test_cancer_fnr", "test_f1_macro"],
                   ascending=[True, True, False])
      .reset_index(drop=True)
)

err_df = pd.DataFrame(errors).reset_index(drop=True)

print(f"OK: {len(res_df)}, Errores: {len(err_df)}")

Sweep mal_clustered: 100%|██████████| 16/16 [18:22<00:00, 68.92s/run, last=27s, avg=51s, elapsed=18m 22s, eta=0s, ok=16, err=0]          

OK: 16, Errores: 0


In [8]:
display(res_df)

,model_name,clf_name,mw,var_quantile,test_cancer_fn,test_cancer_fnr,test_cancer_recall,test_cancer_specificity,test_f1_macro,test_accuracy,test_balanced_accuracy,test_cancer_roc_auc,test_cancer_pr_auc,mlflow_run_id
0,rf_pca0_log0_vq20_mw6p0_malclust,rf,6.0,0.20,30,0.092025,0.907975,0.324138,0.666341,0.709130,0.686544,0.744584,0.862788,8dcb8a72b0d44166b10d6b1c898b986e
1,rf_pca0_log0_vq20_mw3p0_malclust,rf,3.0,0.20,31,0.095092,0.904908,0.427586,0.712776,0.740977,0.719805,0.780241,0.885096,3e3fdecd96cb40c0a0bad9aceee95444
2,rf_pca0_log0_vq20_mw3p0_malclust,rf,3.0,0.20,31,0.095092,0.904908,0.420690,0.707714,0.736730,0.715904,0.777935,0.881838,312ccb392a594dadae95c574e0b67679
3,rf_pca0_log0_vq20_mw6p0_malclust,rf,6.0,0.20,31,0.095092,0.904908,0.351724,0.677692,0.715499,0.694137,0.741115,0.863228,7edc8751070f4b65b46db0b87bd15fe4
4,extratrees_pca0_log0_vq20_mw3p0_malclust,extratrees,3.0,0.20,33,0.101227,0.898773,0.496552,0.739601,0.760085,0.739969,0.811561,0.902213,201b2089637f47d1af7fa5127bcbe0cf
5,rf_pca0_log0_vq15_mw6p0_malclust,rf,6.0,0.15,33,0.101227,0.898773,0.351724,0.675720,0.713376,0.692535,0.738206,0.864289,5430976ddc794c74843e4355922c4228
6,extratrees_pca0_log0_vq15_mw6p0_malclust,extratrees,6.0,0.15,34,0.104294,0.895706,0.441379,0.719479,0.745223,0.725625,0.788460,0.888531,ec7d445d74904146be196dbbe9a49160
7,rf_pca0_log0_vq15_mw3p0_malclust,rf,3.0,0.15,34,0.104294,0.895706,0.441379,0.715809,0.743100,0.722800,0.770901,0.879474,118ab54a74074c2a8e03b5398bd836f6
8,rf_pca0_log0_vq15_mw6p0_malclust,rf,6.0,0.15,34,0.104294,0.895706,0.344828,0.671706,0.709130,0.689856,0.739317,0.864722,a16de9e749934d0fa5d36aa856751a81
9,rf_pca0_log0_vq15_mw3p0_malclust,rf,3.0,0.15,36,0.110429,0.889571,0.427586,0.699618,0.728238,0.707745,0.767009,0.877332,eec14bb416f8478da3ea22c5f58d14cf


In [9]:
if len(err_df) > 0:
    display(err_df)

## Entrenamiento final del mejor modelo

In [10]:
best = res_df.iloc[0].to_dict()
print("Mejor configuracion:")
for k, v in best.items():
    if k != "mlflow_run_id":
        print(f"  {k}: {v}")

Mejor configuracion:
  model_name: rf_pca0_log0_vq20_mw6p0_malclust
  clf_name: rf
  mw: 6.0
  var_quantile: 0.2
  test_cancer_fn: 30
  test_cancer_fnr: 0.09202453987730061
  test_cancer_recall: 0.9079754601226994
  test_cancer_specificity: 0.32413793103448274
  test_f1_macro: 0.6663405259685651
  test_accuracy: 0.7091295116772823
  test_balanced_accuracy: 0.686544455933702
  test_cancer_roc_auc: 0.7445843029405542
  test_cancer_pr_auc: 0.8627880856303899


In [11]:
best_cfg = replace(
    base_cfg,
    model_name="multiclass_mal_clustered_final",
    model_version="v0.1.0",
    clf_name=best["clf_name"],
    malignant_weight=float(best["mw"]),
    var_quantile=float(best["var_quantile"]),
    cv_splits=8,
    save_local_bundle=True,
    save_plots=True,
    output_figures_dir="reports/figures/multiclass_mal_clustered",
)

final_out = run_training(best_cfg, feature_cols=gene_cols)
print(f"\nModelo guardado en: {final_out.get('bundle_dir', 'N/A')}")


Modelo guardado en: /workspaces/TFM/models/multiclass_mal_clustered_final/v0.1.0


## Comparacion con notebook 4.0 (18 tipos de cancer originales)

In [12]:
# Resultados de referencia del notebook 4.0 (mejor modelo)
ref_40 = {
    "cancer_fn": 37,
    "cancer_fnr": 0.1135,
    "cancer_recall": 0.8865,
    "cancer_specificity": 0.5517,
    "f1_macro": 0.2037,
    "accuracy": 0.4565,
    "cancer_roc_auc": 0.8326,
    "cancer_pr_auc": 0.9141,
}

tm = final_out["test_metrics"]
res_41 = {
    "cancer_fn": tm["cancer_fn"],
    "cancer_fnr": round(tm["cancer_fnr"], 4),
    "cancer_recall": round(tm["cancer_recall_sensitivity"], 4),
    "cancer_specificity": round(tm["cancer_specificity"], 4),
    "f1_macro": round(tm["f1_macro"], 4),
    "accuracy": round(tm["accuracy"], 4),
    "cancer_roc_auc": round(tm["cancer_roc_auc"], 4),
    "cancer_pr_auc": round(tm["cancer_pr_auc"], 4),
}

comparison = pd.DataFrame({
    "4.0 (18 tipos cancer)": ref_40,
    "4.1 (clusters malignos)": res_41,
})
comparison["delta"] = comparison["4.1 (clusters malignos)"] - comparison["4.0 (18 tipos cancer)"]

display(comparison)

,4.0 (18 tipos cancer),4.1 (clusters malignos),delta
cancer_fn,37.0000,32.0000,-5.0000
cancer_fnr,0.1135,0.0982,-0.0153
cancer_recall,0.8865,0.9018,0.0153
cancer_specificity,0.5517,0.3517,-0.2000
f1_macro,0.2037,0.6762,0.4725
accuracy,0.4565,0.7134,0.2569
cancer_roc_auc,0.8326,0.7446,-0.0880
cancer_pr_auc,0.9141,0.8628,-0.0513
